# DLinear 추가 실험 (재현 보강)
`DLinear_reproduce.ipynb`로 논문 Table 2를 재현한 뒤, **DLinear에 대해서만** 더 확인하는 노트북.

| 순서 | 실험 | 확인하는 것 | 예상 시간 (T4) |
|---|---|---|---|
| 1 | 시드 편차 | 논문과 차이 난 10칸을 3번씩 돌려 평균·흔들림 확인 | 15분 |
| 2 | 단변량 (논문 Table 9) | 타깃(OT) 하나만 예측하는 설정 | 30분 |
| 3 | 가중치 그림 (논문 Figure 7) | Electricity·Traffic에서 24h·168h 주기가 가중치에 보이는지 | 15분 |
| 4 | `--individual` | 채널마다 W를 따로 둘 때 vs 공유할 때 | 1시간 |
| 5 | 입력 길이 (논문 Figure 4) | L을 48→720으로 늘릴 때 MSE 변화 (Electricity·Traffic, T=720) | 3~4시간 |

**실행:** 서버는 **GPU → T4 → High-RAM** (Electricity·Traffic 테스트 단계에서 메모리를 많이 씀) → **Run All**.
- 하이퍼파라미터는 공식 스크립트(`scripts/EXP-*/*.sh`)에서 그대로 뽑아 쓴다 (`lab.py` 참고).
- 실험마다 한 줄씩 진행 상황이 찍힌다. 중간에 끊겨도 다시 Run All 하면 끝난 실험은 건너뛴다.
- 로그: 서버의 `/content/LTSF-Linear/logs/<실험이름>/`

In [ ]:
# 0. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

In [ ]:
# 1. 코드·패치·데이터 (이미 있으면 건너뜀)
import os, subprocess
os.chdir('/content')
if not os.path.exists('LTSF-Linear'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cure-lab/LTSF-Linear.git'], check=True)
os.chdir('/content/LTSF-Linear')

def patch(path, old, new):
    s = open(path).read()
    if new in s:
        return
    assert old in s, (path, old)
    open(path, 'w').write(s.replace(old, new))

patch('utils/tools.py', 'np.Inf', 'np.inf')                       # NumPy 2 호환 (필수)
patch('run_longExp.py', "default=10, help='data loader num workers'",
      "default=2, help='data loader num workers'")                  # Colab CPU 수에 맞춤 (결과 영향 없음)
# 메모리 절약: test()가 쓰지도 않는 입력 배열(inputx)까지 모아서 L=720 Traffic에서 RAM이 빠듯해진다. 지표 계산과 무관.
patch('exp/exp_main.py', '                inputx.append(batch_x.detach().cpu().numpy())',
      '                pass  # inputx 미사용 (메모리 절약 패치)')
patch('exp/exp_main.py', '        inputx = np.concatenate(inputx, axis=0)', '        inputx = None')

os.makedirs('dataset', exist_ok=True)
for f in ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2']:
    if not os.path.exists(f'dataset/{f}.csv'):
        subprocess.run(['wget', '-q', '-O', f'dataset/{f}.csv',
                        f'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/{f}.csv'], check=True)
need = ['electricity', 'exchange_rate', 'national_illness', 'traffic', 'weather']
if not all(os.path.exists(f'dataset/{f}.csv') for f in need):
    subprocess.run('pip -q install gdown && gdown -q --folder '
                   'https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy -O raw_data && '
                   'find raw_data -name "*.csv" -exec cp -n {} dataset/ \\;', shell=True, check=True)
!ls -lh dataset

In [ ]:
%%writefile /content/lab.py
"""DLinear 추가 실험 도우미.

공식 스크립트(scripts/EXP-*/...sh)를 '가짜 python'으로 한 번 돌려서 실제 실행될 명령 목록만 뽑고,
그 명령을 하나씩 실행한다. 이렇게 하면
  - 하이퍼파라미터를 손으로 옮겨 적지 않고 공식 스크립트 값을 그대로 쓰고
  - 실험마다 진행 상황이 한 줄씩 찍히고
  - 중간에 끊겨도 다시 실행하면 끝난 실험은 건너뛴다.
"""
import json, os, re, shutil, subprocess, sys, time

import pandas as pd

ROOT = '/content/LTSF-Linear'
_SHIM = '/content/_shim'
_WORK = '/content/_collect'

DATASETS = {'etth1.csv': 'ETTh1', 'etth2.csv': 'ETTh2', 'ettm1.csv': 'ETTm1', 'ettm2.csv': 'ETTm2',
            'electricity.csv': 'Electricity', 'exchange_rate.csv': 'Exchange', 'traffic.csv': 'Traffic',
            'weather.csv': 'Weather', 'national_illness.csv': 'ILI'}


def commands(script, edits=()):
    """공식 스크립트에서 실행될 python 명령 목록을 뽑는다. edits = [(찾을 문자열, 바꿀 문자열), ...]"""
    os.makedirs(_SHIM, exist_ok=True)
    with open(f'{_SHIM}/python', 'w') as f:
        f.write('#!/usr/bin/env python3\n'
                'import json, os, sys\n'
                'try:\n    log = os.readlink("/proc/self/fd/1")\nexcept OSError:\n    log = ""\n'
                'with open(os.environ["CMD_FILE"], "a") as f:\n'
                '    f.write(json.dumps({"argv": sys.argv[1:], "log": log}) + "\\n")\n')
    os.chmod(f'{_SHIM}/python', 0o755)

    shutil.rmtree(_WORK, ignore_errors=True)
    for d in ['logs/LongForecasting/univariate', 'logs/LookBackWindow']:  # 스크립트가 안 만드는 폴더도 미리
        os.makedirs(f'{_WORK}/{d}')
    src = open(os.path.join(ROOT, script)).read()
    for a, b in edits:
        assert a in src, f'{a!r} 가 {script} 에 없음'
        src = src.replace(a, b)
    open(f'{_WORK}/run.sh', 'w').write(src)
    env = dict(os.environ, PATH=f'{_SHIM}:{os.environ["PATH"]}', CMD_FILE=f'{_WORK}/cmds.jsonl')
    subprocess.run(['bash', 'run.sh'], cwd=_WORK, env=env, check=True, stdout=subprocess.DEVNULL)
    out = [json.loads(l) for l in open(f'{_WORK}/cmds.jsonl')]
    for c in out:
        c['log'] = os.path.basename(c['log']) if c['log'].startswith(_WORK) else ''
    return out


def arg(c, *names, default=None):
    a = c['argv']
    for i, x in enumerate(a[:-1]):
        if x in names:
            return a[i + 1]
    return default


def set_arg(c, name, value):
    a = c['argv']
    if name in a:
        a[a.index(name) + 1] = str(value)
    else:
        a += [name, str(value)]
    return c


def info(c):
    return dict(model=arg(c, '--model'), dataset=DATASETS[arg(c, '--data_path').lower()],
                features=arg(c, '--features', '--feature', default='M'),
                seq_len=int(arg(c, '--seq_len')), pred_len=int(arg(c, '--pred_len')),
                individual='--individual' in c['argv'])


def parse_log(path):
    """로그에서 (testing 설정, mse, mae)를 순서대로 뽑는다. --itr N이면 N개."""
    if not os.path.exists(path):
        return []
    text = open(path, errors='ignore').read()
    out = []
    for m in re.finditer(r'>>>>>>>testing : (\S+?)<<<.*?mse:([-\d.eE+]+), mae:([-\d.eE+]+)', text, re.S):
        out.append((m.group(1), float(m.group(2)), float(m.group(3))))
    return out


def run(cmds, tag):
    """명령을 차례로 실행. 로그는 logs/<tag>/ 에 저장, 이미 끝난 실험은 건너뜀."""
    logdir = os.path.join(ROOT, 'logs', tag)
    os.makedirs(logdir, exist_ok=True)
    t0 = time.time()
    for i, c in enumerate(cmds, 1):
        c['logpath'] = os.path.join(logdir, c.get('name') or c['log'] or f'run{i}.log')
        d = info(c)
        label = (f"{d['dataset']} {d['features']} L={d['seq_len']} T={d['pred_len']}"
                 + (' individual' if d['individual'] else ''))
        need = int(arg(c, '--itr', default=1))
        if len(parse_log(c['logpath'])) >= need:
            print(f'[{i}/{len(cmds)}] {label}: 이미 완료, 건너뜀', flush=True)
            continue
        t = time.time()
        with open(c['logpath'], 'w') as f:
            r = subprocess.run([sys.executable] + c['argv'], cwd=ROOT, stdout=f, stderr=subprocess.STDOUT)
        subprocess.run('rm -f results/*/pred.npy', shell=True, cwd=ROOT)  # 예측 배열은 안 쓰므로 디스크 절약
        res = parse_log(c['logpath'])
        if r.returncode != 0 or len(res) < need:
            tail = open(c['logpath'], errors='ignore').read()[-600:]
            print(f'[{i}/{len(cmds)}] {label}: 실패 (exit {r.returncode})\n{tail}', flush=True)
            continue
        mses = ', '.join(f'{m:.4f}' for _, m, _ in res)
        print(f'[{i}/{len(cmds)}] {label}: mse {mses}  ({time.time() - t:.0f}s, 누적 {(time.time() - t0) / 60:.0f}분)',
              flush=True)


def collect(cmds):
    rows = []
    for c in cmds:
        for k, (setting, mse, mae) in enumerate(parse_log(c.get('logpath', ''))):
            rows.append(dict(info(c), run=k, mse=mse, mae=mae))
    return pd.DataFrame(rows)

In [ ]:
# 2. 논문 값과 첫 재현 값
import sys, importlib, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '/content'); import lab; importlib.reload(lab)
pd.set_option('display.width', 200)

# 논문 Table 2 (다변량) DLinear MSE, MAE
PAPER = {
 'ETTh1': {96: (0.375, 0.399), 192: (0.405, 0.416), 336: (0.439, 0.443), 720: (0.472, 0.490)},
 'ETTh2': {96: (0.289, 0.353), 192: (0.383, 0.418), 336: (0.448, 0.465), 720: (0.605, 0.551)},
 'ETTm1': {96: (0.299, 0.343), 192: (0.335, 0.365), 336: (0.369, 0.386), 720: (0.425, 0.421)},
 'ETTm2': {96: (0.167, 0.260), 192: (0.224, 0.303), 336: (0.281, 0.342), 720: (0.397, 0.421)},
 'Exchange': {96: (0.081, 0.203), 192: (0.157, 0.293), 336: (0.305, 0.414), 720: (0.643, 0.601)},
 'Weather': {96: (0.176, 0.237), 192: (0.220, 0.282), 336: (0.265, 0.319), 720: (0.323, 0.362)},
 'Electricity': {96: (0.140, 0.237), 192: (0.153, 0.249), 336: (0.169, 0.267), 720: (0.203, 0.301)},
 'Traffic': {96: (0.410, 0.282), 192: (0.423, 0.287), 336: (0.436, 0.296), 720: (0.466, 0.315)},
 'ILI': {24: (2.215, 1.081), 36: (1.963, 0.963), 48: (2.130, 1.024), 60: (2.368, 1.096)},
}
# 논문 Table 9 (단변량, ETT) DLinear MSE, MAE
PAPER_S = {
 'ETTh1': {96: (0.056, 0.180), 192: (0.071, 0.204), 336: (0.098, 0.244), 720: (0.189, 0.359)},
 'ETTh2': {96: (0.131, 0.279), 192: (0.176, 0.329), 336: (0.209, 0.367), 720: (0.276, 0.426)},
 'ETTm1': {96: (0.028, 0.123), 192: (0.045, 0.156), 336: (0.061, 0.182), 720: (0.080, 0.210)},
 'ETTm2': {96: (0.063, 0.183), 192: (0.092, 0.227), 336: (0.119, 0.261), 720: (0.175, 0.320)},
}
# DLinear_reproduce.ipynb 첫 재현 (seed 2021, 1회) MSE
OURS = {
 'ETTh1': {96: 0.384, 192: 0.443, 336: 0.447, 720: 0.504}, 'ETTh2': {96: 0.290, 192: 0.388, 336: 0.463, 720: 0.733},
 'ETTm1': {96: 0.301, 192: 0.336, 336: 0.372, 720: 0.427}, 'ETTm2': {96: 0.172, 192: 0.237, 336: 0.295, 720: 0.427},
 'Exchange': {96: 0.085, 192: 0.162, 336: 0.333, 720: 0.898}, 'Weather': {96: 0.174, 192: 0.218, 336: 0.263, 720: 0.332},
 'Electricity': {96: 0.140, 192: 0.154, 336: 0.169, 720: 0.204}, 'Traffic': {96: 0.413, 192: 0.424, 336: 0.438, 720: 0.466},
 'ILI': {24: 2.280, 36: 2.235, 48: 2.298, 60: 2.573},
}
paper = lambda table, d, t, k=0: table.get(d, {}).get(t, (np.nan, np.nan))[k]
LIN = 'scripts/EXP-LongForecasting/Linear/'
RESULTS = {}

## 실험 1. 시드 편차 — 논문과 차이 난 10칸을 3번씩
첫 재현에서 MSE 차이가 0.02 이상이던 칸만 `--itr 3`으로 다시 돌린다.
`run_longExp.py`는 시작할 때 seed 2021을 한 번만 고정하므로 **run 0은 첫 재현과 같은 값**, run 1·2는 난수 상태가 달라진 상태로 학습한다.

In [ ]:
BAD = {('ETTh1', 192), ('ETTh1', 720), ('ETTh2', 720), ('ETTm2', 720), ('Exchange', 336), ('Exchange', 720),
       ('ILI', 24), ('ILI', 36), ('ILI', 48), ('ILI', 60)}
exp1 = [c for s in ['etth1', 'etth2', 'ettm2', 'exchange_rate', 'ili']
        for c in lab.commands(LIN + s + '.sh', [('--itr 1', '--itr 3')])
        if (lab.info(c)['dataset'], lab.info(c)['pred_len']) in BAD]
lab.run(exp1, 'exp1_seed')

df = lab.collect(exp1); RESULTS['exp1_seed'] = df
t = df.pivot_table(index=['dataset', 'pred_len'], columns='run', values='mse')
t.columns = [f'run{c}' for c in t.columns]
t['mean'] = t.mean(axis=1); t['std'] = t.filter(like='run').std(axis=1)
t['paper'] = [paper(PAPER, d, p) for d, p in t.index]
t['first'] = [OURS[d][p] for d, p in t.index]
t['mean-paper'] = t['mean'] - t['paper']
t['best-paper'] = t.filter(like='run').min(axis=1) - t['paper']
t.round(3)

## 실험 2. 단변량 (논문 Table 9)
공식 `univariate/*.sh` 그대로. 타깃 변수 OT 하나만 입력·예측한다 (`--features S`, `enc_in 1`). ETTh는 T=24·48도 돌지만 논문 표는 96~720만 있다.

In [ ]:
exp2 = [c for s in ['etth1', 'etth2', 'ettm1', 'ettm2'] for c in lab.commands(LIN + 'univariate/' + s + '.sh')]
lab.run(exp2, 'exp2_univariate')

df = lab.collect(exp2); RESULTS['exp2_univariate'] = df
df['paper_mse'] = [paper(PAPER_S, d, p) for d, p in zip(df.dataset, df.pred_len)]
df['paper_mae'] = [paper(PAPER_S, d, p, 1) for d, p in zip(df.dataset, df.pred_len)]
df['mse-paper'] = df['mse'] - df['paper_mse']
df[['dataset', 'pred_len', 'mse', 'mae', 'paper_mse', 'paper_mae', 'mse-paper']].round(3)

## 실험 3. 가중치 그림 (논문 Figure 7)
`models/DLinear.py`에 주석으로 있는 $1/L$ 초기화 두 줄을 잠깐 살려서 Electricity·Traffic(L=336 → T=96)을 학습하고, 학습된 $W_s$(seasonal)·$W_t$(trend)를 그린다. 끝나면 원래 파일로 되돌린다.

In [ ]:
!sed -i 's/^            # self.Linear_\(Seasonal\|Trend\).weight/            self.Linear_\1.weight/' models/DLinear.py
!grep -c "^            self.Linear_.*1/self.seq_len" models/DLinear.py
try:
    exp3 = [c for s in ['electricity', 'traffic'] for c in lab.commands(LIN + s + '.sh') if lab.info(c)['pred_len'] == 96]
    for c in exp3:
        lab.set_arg(c, '--model_id', lab.arg(c, '--model_id') + '_init')
    lab.run(exp3, 'exp3_weights')
finally:
    !git checkout -q models/DLinear.py   # 1/L 초기화 되돌림 (이후 실험은 원래 초기화)
!grep -c "^            # self.Linear_.*1/self.seq_len" models/DLinear.py

import torch
os.makedirs('/content/extra_figs', exist_ok=True)
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for j, c in enumerate(exp3):
    name = lab.info(c)['dataset']
    sd = torch.load(glob.glob(f"checkpoints/{lab.arg(c, '--model_id')}_DLinear*/checkpoint.pth")[0], map_location='cpu')
    for i, k in enumerate(['Linear_Seasonal.weight', 'Linear_Trend.weight']):
        W = sd[k].numpy(); v = np.abs(W).max()
        im = axes[i, j].imshow(W, cmap='RdBu_r', vmin=-v, vmax=v, aspect='auto')
        axes[i, j].set_title(f'{name} {k.split(".")[0]}  (T x L)')
        axes[i, j].set_xlabel('input step (335 = most recent)'); axes[i, j].set_ylabel('output step')
        fig.colorbar(im, ax=axes[i, j])
    w = sd['Linear_Seasonal.weight'].numpy()[0]          # 다음 1시간 예측에 쓰는 가중치
    axes[2, j].plot(w)
    for lag in [24, 48, 72, 168, 336]:
        if lag <= len(w):
            axes[2, j].axvline(len(w) - lag, color='gray', ls=':', lw=1)
    axes[2, j].set_title(f'{name} seasonal weight for output step 1 (dotted: 24·48·72·168·336h before)')
    axes[2, j].set_xlabel('input step')
plt.tight_layout(); plt.savefig('/content/extra_figs/weights_electricity_traffic.png', dpi=120); plt.show()

## 실험 4. `--individual` (채널마다 W 따로)
DLinear 기본값은 모든 채널이 같은 $W$를 공유한다. 공식 DLinear 스크립트에 `--individual`만 붙여서 비교한다.
Electricity(321채널)·Traffic(862채널)은 채널 수만큼 선형층을 for문으로 돌려서 매우 느리고 파라미터가 수억 개가 되어 제외했다.

In [ ]:
exp4 = [c for s in ['etth1', 'etth2', 'ettm1', 'ettm2', 'exchange_rate', 'weather', 'ili']
        for c in lab.commands(LIN + s + '.sh', [('--itr 1', '--itr 1 --individual')])]
lab.run(exp4, 'exp4_individual')

df = lab.collect(exp4); RESULTS['exp4_individual'] = df
df['shared(first)'] = [OURS[d][p] for d, p in zip(df.dataset, df.pred_len)]
df['individual-shared'] = df['mse'] - df['shared(first)']
df['paper'] = [paper(PAPER, d, p) for d, p in zip(df.dataset, df.pred_len)]
df[['dataset', 'pred_len', 'mse', 'shared(first)', 'individual-shared', 'paper']].round(3)

## 실험 5. 입력 길이 (논문 Figure 4)
공식 `Linear_DiffWindow.sh`에서 논문 Figure 4와 같은 조건(Electricity·Traffic, T=720)만 골라 L = 48 … 720으로 돌린다.
선형 모델은 L이 길수록 좋아진다는 게 논문 주장 (Transformer는 오히려 나빠지거나 그대로).

In [ ]:
exp5 = [c for c in lab.commands('scripts/EXP-LookBackWindow/Linear_DiffWindow.sh')
        if lab.info(c)['dataset'] in ('Electricity', 'Traffic') and lab.info(c)['pred_len'] == 720]
lab.run(exp5, 'exp5_lookback')

df = lab.collect(exp5); RESULTS['exp5_lookback'] = df
t = df.pivot_table(index='seq_len', columns='dataset', values='mse')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, d in zip(axes, t.columns):
    ax.plot(t.index, t[d], 'o-'); ax.set_title(f'DLinear {d}, T=720'); ax.set_xlabel('look-back L'); ax.set_ylabel('MSE')
    ax.axvline(336, color='gray', ls=':', lw=1)
plt.tight_layout(); plt.savefig('/content/extra_figs/lookback_electricity_traffic.png', dpi=120); plt.show()
t.round(4)

## 6. 결과 모아 보기
아래 출력을 그대로 복사해서 정리에 쓴다. CSV는 서버 `/content/extra_results/`에도 저장.

In [ ]:
os.makedirs('/content/extra_results', exist_ok=True)
for k, df in RESULTS.items():
    df.to_csv(f'/content/extra_results/{k}.csv', index=False)
    print(f'## {k}  ({len(df)} rows)')
    print(df[['dataset', 'features', 'seq_len', 'pred_len', 'individual', 'run', 'mse', 'mae']]
          .to_csv(index=False, float_format='%.4f'))